# U-Net++ with ResNet-34 Encoder for Cementitious Sorptivity Segmentation

This notebook implements the complete U-Net++ deep learning pipeline for capillary water absorption segmentation in cementitious materials, based on the research framework from:
> **"Automated estimation of cementitious sorptivity via computer vision"** (*Kabir et al., Nature Communications, 2024*) and the comprehensive project investigation plan.

### Key Enhancements & Methodology:
1. **Zero-Leakage Video-Level Dataset Partitioning**:
   - Rather than splitting raw frames independently (which causes cross-partition leakage due to temporally adjacent, nearly identical frames), the dataset of **28 videos (1,250 frames)** is locked strictly at the **video level**.
   - All frames and all 4 annotators' ground-truth annotations belonging to a video remain in the same partition.
   - Partition distribution: **19 Train videos (863 frames, ~69.0%)**, **6 Validation videos (169 frames, ~13.5%)**, and **3 Untouched Test videos (218 frames, ~17.4%)**.
2. **Consensus Mask Ground Truth**:
   - Supplementary Movie 7 in the published research demonstrated noticeable inter-annotator disagreement among the four human annotators.
   - We formulate a canonical consensus mask per frame using pixel-wise majority voting ($\ge 2$ of 4 annotators agree on the wetted region), removing subjective border noise while maintaining individual mask toggling options.
3. **U-Net++ (Nested Dense Skip Pathways) + ResNet-34 Encoder**:
   - ResNet-34 backbone pretrained on ImageNet for high-fidelity multi-scale feature representation.
   - Nested decoder nodes with dense skip connections bridge semantic gaps between encoder and decoder feature maps.
   - Standardized input resolution of **448×448 RGB** and **1 output channel** for binary segmentation.
4. **BCE + Dice Hybrid Loss**:
   - Balanced combination of Binary Cross-Entropy with Logits and Dice Loss to stabilize gradient flow and optimize both pixel-level accuracy and region overlap.
5. **Physical Waterline Post-Processing**:
   - Domain-guided post-processing enforcing bottom-edge contiguity (capillary absorption originates from the bottom reservoir) and column-wise waterline monotonicity to eliminate floating reflections and specular noise.
6. **Downstream Wetted Area Ratio (WAR) Evaluation**:
   - Evaluates not only standard mask overlap metrics (**Dice, IoU, Precision, Recall, Accuracy**) on the untouched test videos (`video 1`, `video 21`, `video 24`), but also calculates downstream **Wetted Area Ratio (WAR)** and reports **WAR MAE, RMSE, and $R^2$**.

## Section 1: Environment Setup, Configuration & Reproducibility
Seeds all random number generators, establishes directory paths, configures GPU acceleration, and defines training hyperparameters.

In [ ]:
import os
import sys
import glob
import time
import random
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import segmentation_models_pytorch as smp

# ==============================================================================
# Global Configuration & Hyperparameters
# ==============================================================================
SEED = 42

def set_seed(seed: int = SEED) -> None:
    """Seeds all random number generators for strict deterministic reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)

# Device Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active compute device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU device name: {torch.cuda.get_device_name(0)}")

# Paths Configuration
DATA_ROOT = Path(r"video-frame/video-frame")
FRAMES_DIR = DATA_ROOT / "frames"
ANNOTATIONS_DIR = DATA_ROOT / "annotations"
CONSENSUS_DIR = DATA_ROOT / "consensus_masks"
SPLIT_CSV_PATH = Path("video_split_assignment.csv")
MODEL_SAVE_DIR = Path("evaluation")
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Training & Image Parameters
IMAGE_SIZE = 448          # 448x448 resolution standard matching FPN benchmark
BATCH_SIZE = 8            # Stable mini-batch size for GPU VRAM
LEARNING_RATE = 1e-4      # Initial learning rate for Adam optimizer
EPOCHS = 50               # Maximum epochs (early stopping will halt if plateaued)
EARLY_STOP_PATIENCE = 6   # Early stopping patience (monitoring validation IoU)
NUM_WORKERS = 0           # 0 to prevent Windows multiprocessing spawn issues

# ImageNet normalization statistics
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Post-processing & Physics Parameters
PRED_THRESHOLD = 0.50          # Probability threshold for binary mask classification
BOTTOM_MARGIN = 32             # Pixel tolerance for bottom reservoir contiguity at 448x448
USE_CONSENSUS_MASKS = True     # Use consensus masks (majority vote) as primary ground truth

print(f"Data Root: {DATA_ROOT.resolve()}")
print(f"Model Save Directory: {MODEL_SAVE_DIR.resolve()}")

## Section 2: Dataset Locking & Video-Level Split Creation
Locks the dataset into **Train**, **Validation**, and **Test** partitions at the **video level**.
- This guarantees zero leakage: all frames and all 4 annotator annotations for a given video remain in the same partition.
- Stores the split assignment in `video_split_assignment.csv` so that FPN, U-Net++, and any other model use the identical partition.

In [ ]:
def lock_video_split(frames_dir: Path, split_csv: Path, seed: int = SEED) -> pd.DataFrame:
    """Generates or loads the locked video-level train/validation/test split.
    
    Args:
        frames_dir: Path to directory containing video frame subfolders (video 1 to video 28).
        split_csv: Path to save/load the video split CSV file.
        seed: Random seed for deterministic video assignment.
        
    Returns:
        DataFrame containing columns: ['video', 'split', 'frames'].
    """
    if split_csv.exists():
        print(f"Loading existing locked video split from: {split_csv}")
        df = pd.read_csv(split_csv)
        return df

    video_folders = sorted(
        [d.name for d in frames_dir.iterdir() if d.is_dir()],
        key=lambda x: int(x.split()[-1]) if x.split()[-1].isdigit() else x
    )
    video_counts = {v: len(list((frames_dir / v).glob("*.jpg"))) for v in video_folders}
    total_frames = sum(video_counts.values())

    random.seed(seed)
    shuffled_vids = list(video_folders)
    random.shuffle(shuffled_vids)

    target_train = 0.70 * total_frames
    target_val = 0.15 * total_frames

    train_vids, val_vids, test_vids = [], [], []
    train_count, val_count, test_count = 0, 0, 0

    for v in shuffled_vids:
        c = video_counts[v]
        if train_count + c <= target_train or len(train_vids) < 19:
            train_vids.append(v)
            train_count += c
        elif val_count + c <= target_val or len(val_vids) < 6:
            val_vids.append(v)
            val_count += c
        else:
            test_vids.append(v)
            test_count += c

    rows = []
    for v in train_vids: rows.append({"video": v, "split": "train", "frames": video_counts[v]})
    for v in val_vids: rows.append({"video": v, "split": "val", "frames": video_counts[v]})
    for v in test_vids: rows.append({"video": v, "split": "test", "frames": video_counts[v]})

    df = pd.DataFrame(rows)
    df.to_csv(split_csv, index=False)
    print(f"Generated new locked video split saved to: {split_csv}")
    return df

split_df = lock_video_split(FRAMES_DIR, SPLIT_CSV_PATH, seed=SEED)
print("\nDataset Summary by Partition:")
summary = split_df.groupby("split")["frames"].agg(["count", "sum"])
summary.columns = ["Videos", "Total Frames"]
summary["Percentage"] = summary["Total Frames"] / summary["Total Frames"].sum() * 100
print(summary.to_string())

test_videos = split_df[split_df["split"] == "test"]["video"].tolist()
print(f"\nUntouched Test Set Videos: {test_videos}")

## Section 3: Consensus Mask Management
Precomputes and caches consensus ground-truth masks.
- For each frame, reads the 4 annotations from `annotator 1` through `annotator 4`.
- Performs pixel-wise majority voting: a pixel is labeled as wet ($1$) if $\ge 2$ of the 4 annotators marked it as wet.
- Saves lossless binary PNG masks to `video-frame/video-frame/consensus_masks/` for high-throughput training.

In [ ]:
def build_consensus_masks(
    frames_dir: Path,
    annotations_dir: Path,
    consensus_dir: Path,
    min_agreement: int = 2
) -> None:
    """Creates consensus masks by aggregating individual annotator annotations.
    
    Args:
        frames_dir: Path containing raw video frames.
        annotations_dir: Path containing individual annotator subfolders.
        consensus_dir: Destination path for cached consensus masks.
        min_agreement: Number of annotators that must agree for a pixel to be marked foreground.
    """
    consensus_dir.mkdir(parents=True, exist_ok=True)
    annotator_names = ["annotator 1", "annotator 2", "annotator 3", "annotator 4"]
    video_folders = sorted(
        [d.name for d in frames_dir.iterdir() if d.is_dir()],
        key=lambda x: int(x.split()[-1]) if x.split()[-1].isdigit() else x
    )

    total_created = 0
    t0 = time.time()
    for vid in video_folders:
        out_vid_dir = consensus_dir / vid
        out_vid_dir.mkdir(parents=True, exist_ok=True)
        frame_files = sorted((frames_dir / vid).glob("*.jpg"))
        for f_path in frame_files:
            stem = f_path.stem
            out_mask_path = out_vid_dir / f"{stem}.png"
            if not out_mask_path.exists():
                ann_masks = []
                for a in annotator_names:
                    ann_p = annotations_dir / a / vid / f"{stem}.png"
                    if ann_p.exists():
                        m_arr = np.array(Image.open(ann_p))
                        # Binary conversion: any non-zero channel indicates foreground
                        is_wet = m_arr.max(axis=-1) > 0 if m_arr.ndim == 3 else m_arr > 0
                        ann_masks.append(is_wet)
                if ann_masks:
                    # Majority vote aggregation
                    stack = np.stack(ann_masks, axis=0)
                    consensus = (stack.sum(axis=0) >= min_agreement).astype(np.uint8) * 255
                    Image.fromarray(consensus).save(out_mask_path)
                    total_created += 1

    t1 = time.time()
    if total_created > 0:
        print(f"Generated {total_created} consensus masks in {t1 - t0:.2f} seconds.")
    else:
        print("All consensus masks are already cached and verified.")

build_consensus_masks(FRAMES_DIR, ANNOTATIONS_DIR, CONSENSUS_DIR, min_agreement=2)

## Section 4: PyTorch Video Dataset & DataLoaders
Defines `SorptivityVideoDataset` with joint spatial and photometric augmentations:
- **Spatial Augmentations**: Random horizontal flip, vertical flip, and random resized crop to cover varying specimen placement.
- **Photometric Augmentations**: Color jitter (brightness and contrast variations) to cover illumination differences (LED reflections).
- **ImageNet Normalization**: Normalizes inputs using ImageNet channel mean and standard deviation for pretrained ResNet-34.

In [ ]:
class SorptivityVideoDataset(Dataset):
    """PyTorch Dataset for paired sorptivity video frames and segmentation masks."""
    def __init__(
        self,
        split: str,
        split_df: pd.DataFrame,
        frames_dir: Path = FRAMES_DIR,
        consensus_dir: Path = CONSENSUS_DIR,
        image_size: int = IMAGE_SIZE,
        is_train: bool = False,
    ):
        self.split = split
        self.image_size = image_size
        self.is_train = is_train

        # Filter videos belonging strictly to this split partition
        videos_in_split = split_df[split_df["split"] == split]["video"].tolist()
        self.samples: List[Tuple[Path, Path, str, str]] = []

        for vid in videos_in_split:
            vid_frames = sorted((frames_dir / vid).glob("*.jpg"))
            for f_path in vid_frames:
                mask_path = consensus_dir / vid / f"{f_path.stem}.png"
                if mask_path.exists():
                    self.samples.append((f_path, mask_path, vid, f_path.stem))

        if len(self.samples) == 0:
            raise RuntimeError(f"No samples found for split: '{split}'")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor, str, str]:
        img_path, mask_path, video_name, stem = self.samples[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # Resize image (bilinear) and mask (nearest neighbor)
        image = TF.resize(image, [self.image_size, self.image_size])
        mask = TF.resize(mask, [self.image_size, self.image_size],
                         interpolation=TF.InterpolationMode.NEAREST)

        # Synchronized Joint Augmentations for Training Split
        if self.is_train:
            # 1. Random Horizontal Flip (physically plausible)
            if random.random() < 0.5:
                image = TF.hflip(image)
                mask = TF.hflip(mask)

            # 2. Random Vertical Flip
            if random.random() < 0.5:
                image = TF.vflip(image)
                mask = TF.vflip(mask)

            # 3. Random Resized Crop / Scale variation
            pad = 24
            image = TF.pad(image, pad)
            mask = TF.pad(mask, pad)
            i, j, h, w = transforms.RandomResizedCrop.get_params(
                image, scale=(0.85, 1.0), ratio=(1.0, 1.0)
            )
            image = TF.resized_crop(image, i, j, h, w, [self.image_size, self.image_size])
            mask = TF.resized_crop(mask, i, j, h, w, [self.image_size, self.image_size],
                                   interpolation=TF.InterpolationMode.NEAREST)

            # 4. Photometric Jitter (brightness & contrast to combat specular reflection noise)
            brightness_factor = random.uniform(0.8, 1.2)
            contrast_factor = random.uniform(0.8, 1.2)
            image = TF.adjust_brightness(image, brightness_factor)
            image = TF.adjust_contrast(image, contrast_factor)

        # Convert to Tensors
        img_tensor = TF.to_tensor(image)  # Shape: (3, H, W), float in [0.0, 1.0]
        img_tensor = TF.normalize(img_tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD)

        mask_np = np.array(mask)
        mask_tensor = torch.from_numpy((mask_np > 127).astype(np.float32)).unsqueeze(0)  # Shape: (1, H, W)

        return img_tensor, mask_tensor, video_name, stem


def get_dataloaders(
    split_df: pd.DataFrame,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Constructs DataLoaders for train, val, and test partitions."""
    train_ds = SorptivityVideoDataset("train", split_df, is_train=True)
    val_ds = SorptivityVideoDataset("val", split_df, is_train=False)
    test_ds = SorptivityVideoDataset("test", split_df, is_train=False)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=False
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=torch.cuda.is_available(), drop_last=False
    )

    print(f"DataLoaders initialized:")
    print(f"  Train: {len(train_ds)} frames ({len(train_loader)} batches)")
    print(f"  Val:   {len(val_ds)} frames ({len(val_loader)} batches)")
    print(f"  Test:  {len(test_ds)} frames ({len(test_loader)} batches)")
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = get_dataloaders(split_df, batch_size=BATCH_SIZE)

## Section 5: U-Net++ Architecture with Pretrained ResNet-34 Encoder
Instantiates the U-Net++ architecture with a ResNet-34 encoder:
- **ResNet-34 Backbone**: 34-layer residual convolutional network initialized with official ImageNet weights.
- **Nested Dense Skip Pathways**: Replaces plain skip connections with a dense convolutional grid ($X^{i,j}$), enabling full aggregation of multi-scale feature maps at different semantic depths.
- **Single Logit Output**: 1 output channel for binary wet-region segmentation.

In [ ]:
def build_unetplusplus_model(
    encoder_name: str = "resnet34",
    encoder_weights: str = "imagenet",
    in_channels: int = 3,
    classes: int = 1
) -> nn.Module:
    """Builds and returns a U-Net++ segmentation model using smp library.
    
    Args:
        encoder_name: Backbone architecture (ResNet-34).
        encoder_weights: Pretrained initialization ('imagenet').
        in_channels: 3 for RGB image inputs.
        classes: 1 for binary segmentation (wet vs dry).
        
    Returns:
        Configured PyTorch nn.Module.
    """
    model = smp.UnetPlusPlus(
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=in_channels,
        classes=classes,
    )
    return model

model = build_unetplusplus_model().to(DEVICE)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net++ (ResNet-34) initialized successfully.")
print(f"Trainable parameters: {num_params:,}")

## Section 6: Loss Function, Evaluation Metrics & Physical Domain Constraints
Defines:
1. **Hybrid BCE + Dice Loss**: Combines pixel-level cross-entropy and area-level Dice overlap.
2. **Segmentation Metrics**: IoU, Dice, Precision, Recall, Accuracy.
3. **Physical Waterline Post-Processing (`enforce_waterline_physics`)**: Eliminates floating noise by enforcing bottom reservoir contiguity and monotonic waterlines.
4. **Wetted Area Ratio (`compute_war`) & Downstream Metrics**: Calculates WAR for predicted and ground-truth masks, reporting MAE, RMSE, and $R^2$.

In [ ]:
class BCEDiceLoss(nn.Module):
    """Hybrid loss combining Binary Cross-Entropy with Logits and Dice Loss."""
    def __init__(self, bce_weight: float = 0.5, dice_weight: float = 0.5, smooth: float = 1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.smooth = smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        intersection = (probs * targets).sum(dim=(1, 2, 3))
        cardinality = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
        dice_loss = 1.0 - (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss.mean()


def compute_batch_metrics(
    preds_binary: torch.Tensor,
    targets_binary: torch.Tensor,
    eps: float = 1e-7
) -> Dict[str, float]:
    """Computes IoU, Dice, Precision, Recall, and Accuracy over a batch of binary masks."""
    tp = (preds_binary * targets_binary).sum(dim=(1, 2, 3))
    fp = (preds_binary * (1.0 - targets_binary)).sum(dim=(1, 2, 3))
    fn = ((1.0 - preds_binary) * targets_binary).sum(dim=(1, 2, 3))
    tn = ((1.0 - preds_binary) * (1.0 - targets_binary)).sum(dim=(1, 2, 3))

    iou = (tp + eps) / (tp + fp + fn + eps)
    dice = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    accuracy = (tp + tn) / (tp + fp + fn + tn + eps)

    return {
        "iou": float(iou.mean().item()),
        "dice": float(dice.mean().item()),
        "precision": float(precision.mean().item()),
        "recall": float(recall.mean().item()),
        "accuracy": float(accuracy.mean().item()),
    }


def enforce_waterline_physics(
    binary_mask: np.ndarray,
    bottom_margin: int = BOTTOM_MARGIN,
    close_kernel_size: Tuple[int, int] = (5, 7),
) -> np.ndarray:
    """Enforces physical sorptivity boundary constraints on a binary mask.
    
    1. Bottom Contiguity: Capillary water enters exclusively from the bottom reservoir.
       Eliminates floating reflection blobs and surface artifacts.
    2. Column Monotonicity: Within each column, only pixels contiguous to the bottom waterline are valid.
    """
    mask = (binary_mask > 0).astype(np.uint8)
    H, W = mask.shape
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return np.zeros_like(mask)

    max_y_reached = max(
        stats[i, cv2.CC_STAT_TOP] + stats[i, cv2.CC_STAT_HEIGHT] - 1
        for i in range(1, num_labels)
    )

    bottom_connected_mask = np.zeros_like(mask)
    for i in range(1, num_labels):
        comp_bottom = stats[i, cv2.CC_STAT_TOP] + stats[i, cv2.CC_STAT_HEIGHT] - 1
        if comp_bottom >= max(H - 1 - bottom_margin, max_y_reached - bottom_margin):
            bottom_connected_mask[labels == i] = 1

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, close_kernel_size)
    closed = cv2.morphologyEx(bottom_connected_mask, cv2.MORPH_CLOSE, kernel)

    final_mask = np.zeros_like(closed)
    for col in range(W):
        wet_rows = np.where(closed[:, col] == 1)[0]
        if len(wet_rows) == 0:
            continue
        bottom_pixel = wet_rows[-1]
        waterline = bottom_pixel
        while waterline >= 0 and closed[waterline, col] == 1:
            waterline -= 1
        waterline += 1
        final_mask[waterline : bottom_pixel + 1, col] = 1

    return final_mask


def compute_war(mask: np.ndarray, roi_mask: Optional[np.ndarray] = None) -> float:
    """Calculates Wetted Area Ratio: |M_wet and M_ROI| / |M_ROI|."""
    if roi_mask is not None:
        roi_count = np.sum(roi_mask > 0)
        return float(np.sum((mask > 0) & (roi_mask > 0)) / max(roi_count, 1))
    return float(np.sum(mask > 0) / max(mask.size, 1))


def compute_war_metrics(pred_wars: np.ndarray, gt_wars: np.ndarray) -> Dict[str, float]:
    """Computes Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and R^2 for WAR."""
    mae = float(np.mean(np.abs(pred_wars - gt_wars)))
    rmse = float(np.sqrt(np.mean((pred_wars - gt_wars) ** 2)))
    ss_res = np.sum((gt_wars - pred_wars) ** 2)
    ss_tot = np.sum((gt_wars - np.mean(gt_wars)) ** 2)
    r2 = float(1.0 - ss_res / max(ss_tot, 1e-9))
    return {
        "war_mae": mae,
        "war_rmse": rmse,
        "war_r2": r2,
    }

criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
print("Loss, optimizer, and metrics initialized.")

## Section 7: Fast Smoke Test
Performs a fast end-to-end check on one train and validation batch to verify tensor shapes, gradient flow, and loss computation without throwing CUDA runtime or dimension errors.

In [ ]:
def run_smoke_test(model: nn.Module, loader: DataLoader, criterion: nn.Module) -> None:
    """Executes a 1-step forward/backward sanity check."""
    model.train()
    images, masks, vids, stems = next(iter(loader))
    images, masks = images.to(DEVICE), masks.to(DEVICE)
    logits = model(images)
    loss = criterion(logits, masks)
    loss.backward()
    optimizer.zero_grad()

    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        metrics = compute_batch_metrics(preds, masks)

    print(f"Smoke test passed successfully!")
    print(f"  Input batch shape : {images.shape}")
    print(f"  Target mask shape : {masks.shape}")
    print(f"  Logits shape      : {logits.shape}")
    print(f"  Batch loss        : {loss.item():.4f}")
    print(f"  Batch IoU         : {metrics['iou']:.4f}")
    print(f"  Batch Dice        : {metrics['dice']:.4f}")

run_smoke_test(model, train_loader, criterion)

## Section 8: Training & Validation Loop Implementation
Executes full model training:
- Tracks Loss, IoU, Dice, Precision, Recall, and Accuracy per epoch.
- Implements **Early Stopping** with patience = 6 monitoring `val_iou`.
- Checkpoints the best model weights to `evaluation/unetpp_best.pth` based on highest validation IoU.
- Logs epoch history to `evaluation/metrics_train_val.csv`.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
) -> Tuple[float, Dict[str, float]]:
    """Trains the model for one complete epoch."""
    model.train()
    total_loss = 0.0
    accumulated_metrics = {"iou": 0.0, "dice": 0.0, "precision": 0.0, "recall": 0.0, "accuracy": 0.0}

    for images, masks, _, _ in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs > PRED_THRESHOLD).float()
            b_metrics = compute_batch_metrics(preds, masks)
            for k in accumulated_metrics:
                accumulated_metrics[k] += b_metrics[k]

    num_batches = len(loader)
    avg_loss = total_loss / max(num_batches, 1)
    avg_metrics = {k: v / max(num_batches, 1) for k, v in accumulated_metrics.items()}
    return avg_loss, avg_metrics


@torch.no_grad()
def evaluate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
) -> Tuple[float, Dict[str, float]]:
    """Evaluates the model across the validation split."""
    model.eval()
    total_loss = 0.0
    accumulated_metrics = {"iou": 0.0, "dice": 0.0, "precision": 0.0, "recall": 0.0, "accuracy": 0.0}

    for images, masks, _, _ in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        logits = model(images)
        loss = criterion(logits, masks)

        total_loss += loss.item()
        probs = torch.sigmoid(logits)
        preds = (probs > PRED_THRESHOLD).float()
        b_metrics = compute_batch_metrics(preds, masks)
        for k in accumulated_metrics:
            accumulated_metrics[k] += b_metrics[k]

    num_batches = len(loader)
    avg_loss = total_loss / max(num_batches, 1)
    avg_metrics = {k: v / max(num_batches, 1) for k, v in accumulated_metrics.items()}
    return avg_loss, avg_metrics


# Execute Training Loop
best_val_iou = -1.0
best_model_path = MODEL_SAVE_DIR / "unetpp_best.pth"
patience_counter = 0
history = []

print(f"Starting U-Net++ training for up to {EPOCHS} epochs on {DEVICE}...")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_m = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_m = evaluate_one_epoch(model, val_loader, criterion)
    scheduler.step(val_m["iou"])
    epoch_time = time.time() - t0

    is_best = val_m["iou"] > best_val_iou
    if is_best:
        best_val_iou = val_m["iou"]
        patience_counter = 0
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_iou": best_val_iou,
            "val_metrics": val_m,
        }, best_model_path)
        save_msg = f" -> Saved best model ({best_model_path.name})"
    else:
        patience_counter += 1
        save_msg = ""

    log_entry = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_iou": train_m["iou"],
        "val_iou": val_m["iou"],
        "train_dice": train_m["dice"],
        "val_dice": val_m["dice"],
        "train_precision": train_m["precision"],
        "val_precision": val_m["precision"],
        "train_recall": train_m["recall"],
        "val_recall": val_m["recall"],
        "time_s": epoch_time,
    }
    history.append(log_entry)

    print(
        f"Epoch {epoch:02d}/{EPOCHS:02d} | "
        f"Train Loss: {train_loss:.4f}  Val Loss: {val_loss:.4f} | "
        f"Train IoU: {train_m['iou']:.4f}  Val IoU: {val_m['iou']:.4f} | "
        f"Train Dice: {train_m['dice']:.4f}  Val Dice: {val_m['dice']:.4f} ({epoch_time:.1f}s){save_msg}"
    )

    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered after {epoch} epochs (patience={EARLY_STOP_PATIENCE}).")
        break

# Save Training Metrics History
train_history_df = pd.DataFrame(history)
train_history_df.to_csv(MODEL_SAVE_DIR / "metrics_train_val.csv", index=False)
print(f"\nBest Validation IoU: {best_val_iou:.4f}")
print(f"Saved epoch metrics log to: {MODEL_SAVE_DIR / 'metrics_train_val.csv'}")

## Section 9: Evaluation on Untouched Test Videos & WAR Computation
Evaluates the best checkpoint on the untouched test videos (`video 1`, `video 21`, `video 24`):
1. Loads the saved best checkpoint (`unetpp_best.pth`).
2. Evaluates standard segmentation metrics: **Dice, IoU, Precision, Recall, Accuracy**.
3. Computes downstream **Wetted Area Ratio (WAR)** for both Ground Truth and Predictions across all test frames.
4. Evaluates physical post-processing (`enforce_waterline_physics`).
5. Reports **WAR MAE, RMSE, and $R^2$**.

In [ ]:
# Load Best Checkpoint
checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print(f"Loaded best checkpoint from epoch {checkpoint.get('epoch', 'N/A')} with Val IoU: {checkpoint.get('best_val_iou', 'N/A'):.4f}")

# Comprehensive Evaluation on Untouched Test Set
test_results = []
gt_wars_raw, pred_wars_raw = [], []
gt_wars_phys, pred_wars_phys = [], []

with torch.no_grad():
    for images, masks, vids, stems in test_loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        logits = model(images)
        probs = torch.sigmoid(logits)
        preds_raw = (probs > PRED_THRESHOLD).float()

        # Batch-level metrics
        b_metrics = compute_batch_metrics(preds_raw, masks)

        preds_raw_np = preds_raw.cpu().numpy()[:, 0]
        masks_np = masks.cpu().numpy()[:, 0]

        for i in range(images.size(0)):
            p_raw = preds_raw_np[i]
            gt = masks_np[i]
            p_phys = enforce_waterline_physics(p_raw, bottom_margin=BOTTOM_MARGIN)

            war_gt = compute_war(gt)
            war_pred_raw = compute_war(p_raw)
            war_pred_phys = compute_war(p_phys)

            gt_wars_raw.append(war_gt)
            pred_wars_raw.append(war_pred_raw)
            gt_wars_phys.append(war_gt)
            pred_wars_phys.append(war_pred_phys)

            test_results.append({
                "video": vids[i],
                "stem": stems[i],
                "war_gt": war_gt,
                "war_pred_raw": war_pred_raw,
                "war_pred_phys": war_pred_phys,
            })

gt_wars_raw_arr = np.array(gt_wars_raw)
pred_wars_raw_arr = np.array(pred_wars_raw)
pred_wars_phys_arr = np.array(pred_wars_phys)

raw_war_metrics = compute_war_metrics(pred_wars_raw_arr, gt_wars_raw_arr)
phys_war_metrics = compute_war_metrics(pred_wars_phys_arr, gt_wars_raw_arr)

# Test Split Segmentation Metrics
_, test_seg_metrics = evaluate_one_epoch(model, test_loader, criterion)

print("=" * 70)
print("             UNTOUCHED TEST SET EVALUATION REPORT (U-Net++)")
print("=" * 70)
print(f"Test Videos Evaluated: {test_videos} ({len(test_results)} frames total)\n")

print("1. Segmentation Overlap Metrics:")
print(f"   - Dice Similarity Coefficient : {test_seg_metrics['dice']:.4f}")
print(f"   - Mean Intersection over Union : {test_seg_metrics['iou']:.4f}")
print(f"   - Precision                   : {test_seg_metrics['precision']:.4f}")
print(f"   - Recall                      : {test_seg_metrics['recall']:.4f}")
print(f"   - Pixel Accuracy              : {test_seg_metrics['accuracy']:.4f}\n")

print("2. Downstream Physical Wetted Area Ratio (WAR) Metrics:")
print("   [Raw Model Output]")
print(f"   - WAR MAE                     : {raw_war_metrics['war_mae']:.4f}")
print(f"   - WAR RMSE                    : {raw_war_metrics['war_rmse']:.4f}")
print(f"   - WAR R^2                      : {raw_war_metrics['war_r2']:.4f}\n")

print("   [Physics-Enforced Waterline Output]")
print(f"   - WAR MAE                     : {phys_war_metrics['war_mae']:.4f}")
print(f"   - WAR RMSE                    : {phys_war_metrics['war_rmse']:.4f}")
print(f"   - WAR R^2                      : {phys_war_metrics['war_r2']:.4f}")
print("=" * 70)

# Save Test Results Summary
test_metrics_summary = [
    {
        "mode": "raw",
        "dice": test_seg_metrics["dice"],
        "iou": test_seg_metrics["iou"],
        "precision": test_seg_metrics["precision"],
        "recall": test_seg_metrics["recall"],
        "accuracy": test_seg_metrics["accuracy"],
        "war_mae": raw_war_metrics["war_mae"],
        "war_rmse": raw_war_metrics["war_rmse"],
        "war_r2": raw_war_metrics["war_r2"],
    },
    {
        "mode": "physics_enforced",
        "dice": test_seg_metrics["dice"],
        "iou": test_seg_metrics["iou"],
        "precision": test_seg_metrics["precision"],
        "recall": test_seg_metrics["recall"],
        "accuracy": test_seg_metrics["accuracy"],
        "war_mae": phys_war_metrics["war_mae"],
        "war_rmse": phys_war_metrics["war_rmse"],
        "war_r2": phys_war_metrics["war_r2"],
    }
]
pd.DataFrame(test_metrics_summary).to_csv(MODEL_SAVE_DIR / "metrics_test.csv", index=False)
pd.DataFrame(test_results).to_csv(MODEL_SAVE_DIR / "test_frame_predictions.csv", index=False)
print(f"Saved test summary to: {MODEL_SAVE_DIR / 'metrics_test.csv'}")
print(f"Saved per-frame test predictions to: {MODEL_SAVE_DIR / 'test_frame_predictions.csv'}")

## Section 10: Qualitative Visualizations & Physical WAR Trajectory Curves
Generates:
1. Multi-panel visual comparisons (Input Image, Ground Truth Consensus Mask, Raw Prediction, Physics-Enforced Prediction, and Error Overlays).
2. Capillary absorption **WAR vs. Frame Index trajectory plots** for each test video, comparing ground truth absorption rise against U-Net++ predictions.

In [ ]:
# Plot 1: Capillary Absorption WAR Trajectory Curves
test_df = pd.DataFrame(test_results)

fig, axes = plt.subplots(1, len(test_videos), figsize=(6 * len(test_videos), 4.5), sharey=True)
if len(test_videos) == 1:
    axes = [axes]

for ax, vid in zip(axes, test_videos):
    vid_data = test_df[test_df["video"] == vid].copy()
    vid_data["frame_num"] = vid_data["stem"].apply(lambda x: int(x.split("-")[-1]) if "-" in x else 0)
    vid_data = vid_data.sort_values("frame_num")

    ax.plot(vid_data["frame_num"], vid_data["war_gt"], label="Ground Truth (Consensus)", color="black", lw=2)
    ax.plot(vid_data["frame_num"], vid_data["war_pred_raw"], label="U-Net++ (Raw)", color="#1f77b4", linestyle="--", lw=1.8)
    ax.plot(vid_data["frame_num"], vid_data["war_pred_phys"], label="U-Net++ (Physics-Enforced)", color="#2ca02c", linestyle="-.", lw=1.8)

    ax.set_title(f"{vid.title()} WAR Trajectory", fontsize=12, fontweight="bold")
    ax.set_xlabel("Video Frame Index", fontsize=10)
    ax.set_ylabel("Wetted Area Ratio (WAR)", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper left", fontsize=8)

plt.tight_layout()
trajectory_plot_path = MODEL_SAVE_DIR / "unetpp_war_trajectories.png"
plt.savefig(trajectory_plot_path, dpi=300)
plt.show()
plt.close(fig)
print(f"Saved WAR trajectory curves to: {trajectory_plot_path}")

# Plot 2: Sample Qualitative Visualizations
sample_indices = [0, len(test_loader.dataset) // 2, len(test_loader.dataset) - 1]
fig, axes = plt.subplots(len(sample_indices), 5, figsize=(18, 3.5 * len(sample_indices)))

for row, idx in enumerate(sample_indices):
    img_t, mask_t, vid_name, stem = test_loader.dataset[idx]
    with torch.no_grad():
        logit = model(img_t.unsqueeze(0).to(DEVICE))
        prob = torch.sigmoid(logit)[0, 0].cpu().numpy()
        p_raw = (prob > PRED_THRESHOLD).astype(np.uint8)
        p_phys = enforce_waterline_physics(p_raw, bottom_margin=BOTTOM_MARGIN)

    gt = mask_t[0].numpy().astype(np.uint8)

    # Denormalize image for display
    img_np = img_t.permute(1, 2, 0).numpy()
    img_disp = np.clip(img_np * IMAGENET_STD + IMAGENET_MEAN, 0, 1)

    # Error Map: Green = True Positive, Red = False Positive, Blue = False Negative
    error_map = np.zeros((*gt.shape, 3), dtype=np.float32)
    error_map[(p_raw == 1) & (gt == 1)] = [0, 1, 0]  # TP (Green)
    error_map[(p_raw == 1) & (gt == 0)] = [1, 0, 0]  # FP (Red)
    error_map[(p_raw == 0) & (gt == 1)] = [0, 0, 1]  # FN (Blue)

    axes[row, 0].imshow(img_disp); axes[row, 0].set_title(f"{vid_name} - {stem}\\nInput Frame")
    axes[row, 1].imshow(gt, cmap="gray"); axes[row, 1].set_title(f"Ground Truth\\nWAR: {compute_war(gt):.3f}")
    axes[row, 2].imshow(p_raw, cmap="gray"); axes[row, 2].set_title(f"Raw U-Net++\\nWAR: {compute_war(p_raw):.3f}")
    axes[row, 3].imshow(p_phys, cmap="gray"); axes[row, 3].set_title(f"Physics Enforced\\nWAR: {compute_war(p_phys):.3f}")
    axes[row, 4].imshow(error_map); axes[row, 4].set_title("Error Map\\n(G:TP, R:FP, B:FN)")

    for c in range(5):
        axes[row, c].axis("off")

plt.tight_layout()
visual_comparison_path = MODEL_SAVE_DIR / "unetpp_visual_predictions.png"
plt.savefig(visual_comparison_path, dpi=300)
plt.show()
plt.close(fig)
print(f"Saved visual prediction panels to: {visual_comparison_path}")